In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
import os
import sys
from pathlib import Path

library_path = os.path.abspath('../src')
if library_path not in sys.path:
    sys.path.append(library_path)
library_path = Path(library_path)
library_path

In [ ]:
# load data
DATA_PATH = library_path.parent / "data"
PLOTS_PATH = library_path.parent / "plots"

cols_to_use = ["event", "months"]

df = pd.read_csv(f"{DATA_PATH}/GPT_processed_survival_data.csv", usecols=cols_to_use)

In [ ]:
df['months_round'] = df['months'].round().astype(int)
df.info()


In [ ]:
df.head()

In [ ]:
from lifelines import KaplanMeierFitter

In [ ]:
kmf = KaplanMeierFitter()

# ── 1. Overall survival curve ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
kmf.fit(df["months_round"], event_observed=df["event"], label="All patients")
kmf.plot_survival_function(ax=ax, ci_show=True)
ax.set_title("Overall Survival — All Patients")
ax.set_xlabel("Time (months)")
ax.set_ylabel("Survival Probability")
median_os = kmf.median_survival_time_
ax.axhline(0.5, color="grey", linestyle="--", alpha=0.5)
ax.text(0, 0.52, f"Median OS: {median_os:.1f} months", fontsize=9, color="grey")
plt.tight_layout()
plt.show()

In [ ]:
kmf.event_table.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
kmf.fit(df["months_round"], event_observed=df["event"], label="All patients")
kmf.plot_survival_function(ax=ax, ci_show=True, at_risk_counts=True)
ax.set_title("Overall Survival — All Patients")
ax.set_xlabel("Time (months)")
ax.set_ylabel("Survival Probability")
median_os = kmf.median_survival_time_
ax.axhline(0.5, color="grey", linestyle="--", alpha=0.5)
ax.text(0, 0.52, f"Median OS: {median_os:.1f} months", fontsize=9, color="grey")
plt.tight_layout()
plt.savefig(f"{PLOTS_PATH}/overall_survival_with_at_risk_counts.png", dpi=600)
plt.show()

In [ ]:
def cutoff_time_from_at_risk(kmf_obj, min_at_risk):
    t = kmf_obj.event_table.query("at_risk < @min_at_risk").index.min()
    if pd.isna(t):
        t = kmf_obj.event_table.index.max()
    return float(t)

In [ ]:
n_patients = len(df)
min_at_risk_primary = int(np.ceil(0.10 * n_patients))
t_cutoff = cutoff_time_from_at_risk(kmf, min_at_risk_primary)

print(f"Primary cutoff rule: <10% at risk (n < {min_at_risk_primary})")
print(f"Selected tau: {t_cutoff:.1f} months")

In [ ]:
from lifelines.utils import restricted_mean_survival_time

rmst = restricted_mean_survival_time(kmf, t=t_cutoff)
rmst

In [ ]:
def bootstrap_rmst(time, event, t, n_bootstraps=1000, ci=0.95, random_state=None):
    """
    Bootstrap the Restricted Mean Survival Time (RMST).

    Parameters
    ----------
    time : array-like
        Observed survival / follow-up times.
    event : array-like
        Event indicator (1 = event occurred, 0 = censored).
    t : float
        Restriction time horizon for the RMST.
    n_bootstraps : int
        Number of bootstrap resamples.
    ci : float
        Coverage of the confidence interval (default 0.95).
    random_state : int or None
        Seed for reproducibility.

    Returns
    -------
    dict with keys:
        rmst_observed  – RMST on the original data
        rmst_bootstrap – array of bootstrap RMST values
        ci_lower       – lower bound of the bootstrap percentile CI
        ci_upper       – upper bound of the bootstrap percentile CI
        se             – bootstrap standard error
    """
    rng = np.random.default_rng(random_state)
    time = np.asarray(time)
    event = np.asarray(event)
    n = len(time)

    # RMST on the original sample
    _kmf = KaplanMeierFitter()
    _kmf.fit(time, event_observed=event)
    rmst_obs = restricted_mean_survival_time(_kmf, t=t)

    # Bootstrap loop
    rmst_boot = np.empty(n_bootstraps)
    for i in range(n_bootstraps):
        idx = rng.integers(0, n, size=n)
        _kmf_b = KaplanMeierFitter()
        _kmf_b.fit(time[idx], event_observed=event[idx])
        rmst_boot[i] = restricted_mean_survival_time(_kmf_b, t=t)

    alpha = 1 - ci
    ci_lower = np.percentile(rmst_boot, 100 * alpha / 2)
    ci_upper = np.percentile(rmst_boot, 100 * (1 - alpha / 2))

    return {
        "rmst_observed": rmst_obs,
        "rmst_bootstrap": rmst_boot,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "se": rmst_boot.std(),
    }


In [ ]:
result = bootstrap_rmst(df["months_round"], df["event"], t=t_cutoff, n_bootstraps=1000, random_state=42)

print(f"RMST (observed):  {result['rmst_observed']:.2f} months")
print(f"Bootstrap SE:     {result['se']:.2f} months")
print(f"95% CI:           [{result['ci_lower']:.2f}, {result['ci_upper']:.2f}] months")


In [ ]:
# Sensitivity analysis for alternative at-risk cutoff rules



n_patients = len(df)

rules = [
    {"rule": "<10 at risk", "min_at_risk": 10},
    {"rule": "<15 at risk", "min_at_risk": 15},
    {"rule": "<20 at risk", "min_at_risk": 20},
    {"rule": "<10% at risk", "min_at_risk": int(np.ceil(0.10 * n_patients))},
]

rows = []
for r in rules:
    tau = cutoff_time_from_at_risk(kmf, r["min_at_risk"])
    boot = bootstrap_rmst(
        df["months_round"],
        df["event"],
        t=tau,
        n_bootstraps=1000,
        random_state=42
    )
    rows.append({
        "rule": r["rule"],
        "threshold_n": r["min_at_risk"],
        "tau_months": tau,
        "rmst_months": boot["rmst_observed"],
        "ci_lower": boot["ci_lower"],
        "ci_upper": boot["ci_upper"],
        "se": boot["se"],
    })

sensitivity_df = pd.DataFrame(rows)

primary_rule = "<10% at risk"
primary_rmst = sensitivity_df.loc[sensitivity_df["rule"] == primary_rule, "rmst_months"].iloc[0]
sensitivity_df["delta_vs_primary"] = sensitivity_df["rmst_months"] - primary_rmst

sensitivity_df.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

y = sensitivity_df["rmst_months"].values
yerr = np.vstack([
    y - sensitivity_df["ci_lower"].values,
    sensitivity_df["ci_upper"].values - y,
])

ax.errorbar(
    sensitivity_df["rule"],
    y,
    yerr=yerr,
    fmt="o-",
    capsize=4,
    color="navy"
)

ax.set_title("Sensitivity Analysis of RMST Truncation Rule")
ax.set_xlabel("Cutoff rule")
ax.set_ylabel("RMST (months)")
ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
for _, row in sensitivity_df.iterrows():
    print(
        f"{row['rule']}: tau={row['tau_months']:.1f} months, "
        f"RMST={row['rmst_months']:.2f} "
        f"[{row['ci_lower']:.2f}, {row['ci_upper']:.2f}], "
        f"Δ vs primary={row['delta_vs_primary']:+.2f}"
    )

In [ ]:
df["months_trunc"] = df["months_round"].clip(upper=t_cutoff)
df["event_trunc"] = ((df["event"] == 1) & (df["months_round"] <= t_cutoff)).astype(int)

In [ ]:
kmf_trunc = KaplanMeierFitter()
kmf_trunc.fit(df["months_trunc"], event_observed=df["event_trunc"], label="All patients")

fig, ax = plt.subplots(figsize=(9, 6))
kmf_trunc.plot_survival_function(ax=ax, ci_show=True, at_risk_counts=True)
ax.set_title("Overall Survival — All Patients")
ax.set_xlabel("Time (months)")
ax.set_ylabel("Survival Probability")
median_os = kmf_trunc.median_survival_time_
ax.axhline(0.5, color="grey", linestyle="--", alpha=0.5)
ax.text(0, 0.52, f"Median OS: {median_os:.1f} months", fontsize=9, color="grey")
plt.tight_layout()
plt.show()

The primary RMST truncation time was defined as the first time point at which fewer than 10% of patients remained at risk, to reduce instability in the tail of the Kaplan–Meier curve. Fixed-number thresholds were evaluated in sensitivity analyses.